In [ ]:
# Import config
from pathlib import Path
import sys

p = Path.cwd()
for _ in range(6):
    if (p / "dir_config.py").exists():
        sys.path.insert(0, str(p))
        break
    p = p.parent

from dir_config import *

# create main dirs + language subfolders
ensure_dirs(
    DATA_DIR, RAW_DIR, RESULT_DIR, MODELS_DIR, LOG_DIR, CHARACTER_DIR, TEST_DATA_DIR
)
ensure_dirs(*(RAW_DIR / lang for lang in LANGUAGES))
ensure_dirs(*(RESULT_DIR / lang for lang in LANGUAGES))

# Code

Check GPU and set device

In [6]:
import torch

if torch.cuda.is_available():
    print(f"torch.version.cuda: {torch.version.cuda}")
    print(f"torch.backends.cudnn.enabled: {torch.backends.cudnn.enabled}")
    !nvidia-smi
    !nvcc --version
else:
    print("GPU is not available")


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on {device}")

torch.version.cuda: 12.9
torch.backends.cudnn.enabled: True
Tue Mar 24 01:02:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.88                 Driver Version: 580.88         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P8              1W /  125W |    1478MiB /   8188MiB |      0%      Default |
|                                         |                        |            

## Load MAGI model

In [2]:
from transformers import AutoModel
import torch

model = (
    AutoModel.from_pretrained("ragavsachdeva/magiv2", trust_remote_code=True)
    .cuda()
    .eval()
)

d:\miniconda3\envs\master-thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
d:\miniconda3\envs\master-thesis\Lib\site-packages\torch\nn\modules\module.py:2441: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
d:\miniconda3\envs\maste

## Save/Load model to local

In [ ]:
# Save model to directory:
import os

magi_model_dir = os.path.join(MODELS_DIR, "magi_model_v2")
os.makedirs(magi_model_dir, exist_ok=True)
model.save_pretrained(magi_model_dir)

In [ ]:
# Load model from directory:
from transformers import AutoModel

model = (
    AutoModel.from_pretrained(
        os.path.join(MODELS_DIR, "magi_model_v2"), trust_remote_code=True
    )
    .cuda()
    .eval()
);

d:\miniconda3\envs\master-thesis-magi-v2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'MODELS_DIR' is not defined

In [ ]:
####################### OR ###########################
# Cache model to directory:
# model_1 = AutoModel.from_pretrained("ragavsachdeva/magiv2", cache_dir="./magi_saved_model_directory/", trust_remote_code=True).cuda().eval()

# Code

In [3]:
import os
from pathlib import Path

data_folder = TEST_DATA_DIR
languages_folder = os.listdir(data_folder)
chapter_folder_list = []

for language in languages_folder:
    language_path = os.path.join(data_folder, language)

    for manga in os.listdir(language_path):
        manga_path = os.path.join(language_path, manga)
        base_manga_path = Path(manga_path)

        for chapter_and_db_path in base_manga_path.iterdir():

            # Filter out the db file when crawling, only process with chapter folder
            if chapter_and_db_path.is_dir():
                chapter_folder_list.append(chapter_and_db_path)

print(chapter_folder_list)

[WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Almark/Vol. 1 Ch. 1'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Fami-res Iko/Vol. 1 Ch. 0'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Itsuka Fukushuu Suru Sono Tame ni/Ch. 1'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Itsuka Fukushuu Suru Sono Tame ni/Ch. 2'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Itsuka Fukushuu Suru Sono Tame ni/Ch. 3'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Toukyou KINOKO Sekai Ranking 1-i no Komyu-ryoku Saijaku JK/Ch. 1'), WindowsPath('D:/Downloads/Tu_Lieu/Cao_Hoc/Master_Thesis/master-thesis/data/test_data/en/Yuusha Party kara Oidasareta Fuguushoku [Wanashi] Yunikku Sukiru [Yajirushi] de Saikyou ni Naru/Ch. 1'), WindowsPath('D:/Downloads/Tu_

# Define data/result directory

In [ ]:
import os

data_folder = TEST_DATA_DIR
result_folder = RESULT_DIR
language = LANGUAGES[1]  # Pick vi
manga_name = "Almark"
chapter_name = "Vol. 1 Ch. 1"
manga_image_list = os.path.join(
    language, manga_name, chapter_name
)  # Will be changed to list later
character_folder = CHARACTER_DIR

# Almark
input_magiv2_folder = os.path.join(data_folder, manga_image_list)
output_magiv2_folder = os.path.join(result_folder, manga_image_list)
json_output_dir = os.path.join(output_magiv2_folder, JSON_RESULTS)
result_image_output_dir = os.path.join(output_magiv2_folder, MAGI_IMAGE_RESULT)
os.makedirs(json_output_dir, exist_ok=True)  # Create the directory if it doesn't exist
os.makedirs(
    result_image_output_dir, exist_ok=True
)  # Create the directory if it doesn't exist

## Create raw and character/names list

In [ ]:
import os
import re


def create_chapter_pages_and_character_bank(input_magiv2_folder, character_folder):
    # Create lists for chapter pages and character bank
    chapter_pages = []
    character_bank = {"images": [], "names": []}

    #     Iterate through manga images to create chapter_pages
    for image_file in os.listdir(input_magiv2_folder):
        if image_file.endswith(
            (".png", ".jpg", ".jpeg")
        ):  # Check for image file extensions
            # Extract the page number using regex
            match = re.search(r"p(\d+)", image_file)
            if match:
                page_number = int(match.group(1))  # Convert to integer for sorting
                chapter_pages.append(
                    (page_number, image_file)
                )  # Store as tuple (page_number, image_file)
            else:
                page_number = image_file.rsplit(".", 1)[0]
                chapter_pages.append(
                    (page_number, image_file)
                )  # Store as tuple (page_number, image_file)

    # Sort chapter pages by page number
    chapter_pages.sort(key=lambda x: x[0])
    chapter_pages = [
        os.path.join(input_magiv2_folder, img[1]) for img in chapter_pages
    ]  # Extract just the filenames after sorting

    # Iterate through character images to create character bank
    for char_image_file in os.listdir(character_folder):
        if char_image_file.endswith(
            (".png", ".jpg", ".jpeg")
        ):  # Check for image file extensions
            # Split the filename to extract character name
            char_name = char_image_file.split("_")[
                0
            ]  # Get the part before the underscore
            character_bank["images"].append(
                os.path.join(character_folder, char_image_file)
            )
            character_bank["names"].append(char_name)
    return chapter_pages, character_bank


# Get chapter pages and character bank
chapter_pages_original, character_bank_original = (
    create_chapter_pages_and_character_bank(input_magiv2_folder, character_folder)
)

chapter_pages_test = chapter_pages_original[:]
character_bank_test = character_bank_original

# Print the results (for debugging)
print("Chapter Pages:")
print(chapter_pages_test)

print("\nCharacter Bank:")
print(character_bank_test)

Chapter Pages:
['D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\00.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\01.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\02.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\03.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\04.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\05.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\06.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\test_data\\vi/Almark/Vol. 1 Ch. 1\\07.jpg', 'D:\\Downloads\\Tu_Lieu\\Cao_Hoc\\Master_Thesis\\master-thesis\\data\\te

## Process (OCR → Transcript)

In [ ]:
from PIL import Image
import numpy as np


def read_image(path_to_image):
    with open(path_to_image, "rb") as file:
        image = Image.open(file).convert("L").convert("RGB")
        image = np.array(image)
    return image


chapter_pages = [read_image(x) for x in chapter_pages_test]
character_bank = character_bank_test.copy()
character_bank["images"] = [read_image(x) for x in character_bank_test["images"]]

with torch.no_grad():
    per_page_results = model.do_chapter_wide_prediction(
        chapter_pages, character_bank, use_tqdm=True, do_ocr=True
    )

print("Continue with next cell")

100%|██████████| 16/16 [03:35<00:00, 13.46s/it]

Continue with next cell


## Save transcript

In [ ]:
transcript = []
for i, (image, page_result) in enumerate(zip(chapter_pages, per_page_results)):
    image_name_ext = os.path.basename(chapter_pages_test[i])
    # Split the image name and its extension
    image_name, image_extension = os.path.splitext(image_name_ext)

    model.visualise_single_image_prediction(
        image, page_result, os.path.join(result_image_output_dir, f"{image_name}.png")
    )
    # Save page_result to JSON
    json_file_path = os.path.join(
        json_output_dir, f"{image_name}.json"
    )  # Create full file path
    with open(json_file_path, "w") as json_file:
        json.dump(page_result, json_file, indent=4)  # Save with pretty printing

print("\n\nDone you WEEEEB!")

NameError: name 'chapter_pages' is not defined